# SATF-CNN + DANN UDA for two-session SSVEP

在上一版 SATF-CNN 论文结构适配代码上，只增加 DANN 域分类器与 GRL 对抗学习。

实验角色固定为：

```text
source session: test
  test_2 ~ test_7  -> 有标签训练
  source validation -> 只从 test_2~7 内部切分
  test_8           -> 冻结后 source diagnostic

target session: four
  four_1 ~ four_4  -> 只读取 EEG，无标签 UDA adaptation
  four_5           -> 冻结后 final target test
```

训练目标：

```text
SATF-CNN backbone -> 16-D fused feature
                    ├─ frequency head 16->3
                    │    只对 source(test) 计算分类 CE
                    └─ GRL -> domain head 16->32->2
                         source(test)=0 / target(four)=1
                         source + target 都进入域 CE
```

因此这是标准 **two-domain UDA DANN**：目标 session 的无标签 EEG 参与训练，但 `four_5` 的 EEG/标签在模型冻结前都不可见。

SATF-CNN 本体、论文已公开结构及此前标出的论文歧义保持不变。

In [15]:
from pathlib import Path
import math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.signal import resample
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, recall_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

device: cpu


## 1. Two-session UDA protocol: test=source, four=target

In [16]:
DATA_DIR = Path("data")

# Source session: labeled
# TRAIN_FILES = [f"test_{i}_ssvep" for i in range(2, 8)]
# TEST_FILES = ["test_8_ssvep"]

# # Target session: unlabeled adaptation + locked final test
# ADAPTATION_FILES = [f"four_{i}_ssvep" for i in range(1, 5)]
# DRIFT_FILES = ["four_5_ssvep"]

TRAIN_FILES = [f"four_{i}_ssvep" for i in range(1, 5)]
TEST_FILES = ["four_5_ssvep"]

# Target session: unlabeled adaptation + locked final test
ADAPTATION_FILES = [f"test_{i}_ssvep" for i in range(2, 8)]
DRIFT_FILES = ["test_8_ssvep"]


FREQUENCIES_HZ = np.array([15.0, 12.0, 10.0])
NUM_CLASSES = len(FREQUENCIES_HZ)

SAMPLE_RATE_HZ = 250.0
FOUR_DELAY_SECONDS = 0.5
FOUR_WINDOW_SECONDS = 1.5
MODEL_SAMPLES = 372
EEG_CHANNELS = 8
VALIDATION_FRACTION = 0.20

BATCH_SIZE = 64
SOURCE_BATCH_SIZE = BATCH_SIZE // 2
TARGET_BATCH_SIZE = BATCH_SIZE // 2
MAX_EPOCHS = 500
LEARNING_RATE = 1e-3
LR_PATIENCE = 20
EARLY_STOP_PATIENCE = 60

# DANN: GRL strength itself ramps 0 -> 1 during training.
# Domain CE keeps unit weight; no target class labels are used.
DANN_DOMAIN_WEIGHT = 1.0
DANN_GRL_GAMMA = 10.0

TOPK_RATIOS = (0.50, 2 / 3, 0.75)
KAN_GRID_SIZE = 5
KAN_SPLINE_ORDER = 3

roles = TRAIN_FILES + TEST_FILES + ADAPTATION_FILES + DRIFT_FILES
assert len(roles) == len(set(roles)), "file leakage across roles"
assert set(TRAIN_FILES).isdisjoint(TEST_FILES + ADAPTATION_FILES + DRIFT_FILES)
assert set(TEST_FILES).isdisjoint(ADAPTATION_FILES + DRIFT_FILES)
assert set(ADAPTATION_FILES).isdisjoint(DRIFT_FILES)

MODEL_FROZEN = False

print("TWO-SESSION UDA FILE ROLES")
print("  source labeled train/internal val:", TRAIN_FILES)
print("  source diagnostic locked until freeze:", TEST_FILES)
print("  target unlabeled adaptation:", ADAPTATION_FILES)
print("  target final test locked until freeze:", DRIFT_FILES)
print("  source domain=test, target domain=four")

TWO-SESSION UDA FILE ROLES
  source labeled train/internal val: ['four_1_ssvep', 'four_2_ssvep', 'four_3_ssvep', 'four_4_ssvep']
  source diagnostic locked until freeze: ['four_5_ssvep']
  target unlabeled adaptation: ['test_2_ssvep', 'test_3_ssvep', 'test_4_ssvep', 'test_5_ssvep', 'test_6_ssvep', 'test_7_ssvep']
  target final test locked until freeze: ['test_8_ssvep']
  source domain=test, target domain=four


In [17]:
def prepare_trials(raw, name):
    x = np.stack(raw) if np.asarray(raw).dtype == object else np.asarray(raw)
    if x.ndim != 3 or x.shape[1] != EEG_CHANNELS:
        raise ValueError(f"{name}: expected [trial,{EEG_CHANNELS},time], got {x.shape}")

    if name.startswith("four_"):
        start = round(FOUR_DELAY_SECONDS * SAMPLE_RATE_HZ)
        requested = round(FOUR_WINDOW_SECONDS * SAMPLE_RATE_HZ)
        if x.shape[-1] <= start:
            raise ValueError(f"{name}: cannot skip {start} samples")
        x = x[:, :, start:min(start + requested, x.shape[-1])]
    elif not name.startswith("test_"):
        raise ValueError(f"{name}: unknown session prefix")

    original_samples = x.shape[-1]
    if original_samples != MODEL_SAMPLES:
        x = resample(x, MODEL_SAMPLES, axis=-1)

    x = np.asarray(x, dtype=np.float32)
    mean = x.mean(axis=-1, keepdims=True)
    std = x.std(axis=-1, keepdims=True)
    x = (x - mean) / np.maximum(std, 1e-6)
    return x, original_samples


def load_labeled_files(file_names):
    # Before freeze, only source TRAIN_FILES may be read with labels.
    if not MODEL_FROZEN:
        if not set(file_names).issubset(set(TRAIN_FILES)):
            raise RuntimeError(
                "Before freeze, labeled loader may only read source TRAIN_FILES."
            )
    else:
        allowed = set(TEST_FILES + DRIFT_FILES)
        if not set(file_names).issubset(allowed):
            raise RuntimeError("After freeze, final labeled loader only accepts TEST/DRIFT files.")

    records = []
    for name in file_names:
        x, original_samples = prepare_trials(
            np.load(DATA_DIR / f"eeg_trials_{name}.npy", allow_pickle=True), name
        )
        y = np.load(DATA_DIR / f"eeg_labels_{name}.npy").astype(np.int64)

        if len(x) != len(y):
            raise ValueError(f"{name}: X/y mismatch")
        if not np.isin(y, np.arange(NUM_CLASSES)).all():
            raise ValueError(f"{name}: labels must be 0..{NUM_CLASSES-1}")

        records.append((x, y, name))
        print("labeled", name, "original", original_samples,
              "shape", x.shape, "counts", np.bincount(y, minlength=NUM_CLASSES))
    return records


def load_target_unlabeled_files(file_names):
    # Critical UDA rule: this function has no label path at all.
    if MODEL_FROZEN:
        raise RuntimeError("Target adaptation loader is training-only.")
    if not set(file_names).issubset(set(ADAPTATION_FILES)):
        raise RuntimeError("Unlabeled target loader may only read ADAPTATION_FILES.")

    records = []
    for name in file_names:
        x, original_samples = prepare_trials(
            np.load(DATA_DIR / f"eeg_trials_{name}.npy", allow_pickle=True), name
        )
        records.append((x, name))
        print("UNLABELED target", name, "original", original_samples, "shape", x.shape)
    return records


def concatenate_labeled(records):
    return (
        np.concatenate([x for x, _, _ in records]),
        np.concatenate([y for _, y, _ in records]),
        np.concatenate([np.full(len(y), name, dtype=object) for _, y, name in records]),
    )


def concatenate_unlabeled(records):
    return (
        np.concatenate([x for x, _ in records]),
        np.concatenate([np.full(len(x), name, dtype=object) for x, name in records]),
    )

In [18]:
# Source labeled data
source_records = load_labeled_files(TRAIN_FILES)
X_pool, y_pool, source_pool = concatenate_labeled(source_records)

idx = np.arange(len(y_pool))
strata = np.array([f"{s}::{y}" for s, y in zip(source_pool, y_pool)])

train_idx, val_idx = train_test_split(
    idx,
    test_size=VALIDATION_FRACTION,
    random_state=SEED,
    stratify=strata,
)
assert set(train_idx).isdisjoint(val_idx)

X_train, y_train = X_pool[train_idx], y_pool[train_idx]
X_validation, y_validation = X_pool[val_idx], y_pool[val_idx]

# Target session UDA: EEG ONLY, no labels loaded.
target_records = load_target_unlabeled_files(ADAPTATION_FILES)
X_target_adapt, target_source = concatenate_unlabeled(target_records)

print("source train / validation:", len(y_train), len(y_validation))
print("target unlabeled adaptation:", len(X_target_adapt))
print("test_8/four_5 remain LOCKED")

labeled four_1_ssvep original 373 shape (46, 8, 372) counts [22  8 16]
labeled four_2_ssvep original 372 shape (152, 8, 372) counts [38 84 30]
labeled four_3_ssvep original 372 shape (95, 8, 372) counts [33 31 31]
labeled four_4_ssvep original 372 shape (122, 8, 372) counts [55 27 40]
UNLABELED target test_2_ssvep original 374 shape (19, 8, 372)
UNLABELED target test_3_ssvep original 373 shape (41, 8, 372)
UNLABELED target test_4_ssvep original 374 shape (69, 8, 372)
UNLABELED target test_5_ssvep original 373 shape (102, 8, 372)
UNLABELED target test_6_ssvep original 373 shape (106, 8, 372)
UNLABELED target test_7_ssvep original 373 shape (67, 8, 372)
source train / validation: 332 83
target unlabeled adaptation: 404
test_8/four_5 remain LOCKED


## 2. Paper sinusoidal position encoding

In [19]:
class PaperSinusoidalPositionEncoding(nn.Module):
    def __init__(self, channels, samples):
        super().__init__()
        pe = torch.zeros(1, 1, channels, samples, dtype=torch.float32)
        position = torch.arange(samples, dtype=torch.float32)

        pair_index = torch.arange((channels + 1)//2, dtype=torch.float32)
        denominator = torch.pow(10000.0, (2.0 * pair_index) / channels)

        for c in range(channels):
            i = c // 2
            phase = position / denominator[i]
            pe[0, 0, c] = torch.sin(phase) if c % 2 == 0 else torch.cos(phase)

        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe

## 3. PESA: 1×1 Conv + 3×3 DW-Conv, 3 heads, channel Top-k

In [20]:
class QKVProjection(nn.Module):
    def __init__(self, heads):
        super().__init__()
        self.pointwise = nn.Conv2d(1, heads, kernel_size=1, bias=False)
        self.depthwise = nn.Conv2d(
            heads, heads, kernel_size=3, padding=1, groups=heads, bias=False
        )

    def forward(self, x):
        return self.depthwise(self.pointwise(x))


class PaperPESA(nn.Module):
    def __init__(self, channels, heads=3, ratios=TOPK_RATIOS):
        super().__init__()
        self.channels = channels
        self.heads = heads

        # PAPER_UNSPECIFIED_ASSUMPTION: floor rounding
        self.k_values = tuple(
            max(1, min(channels, math.floor(channels * ratio)))
            for ratio in ratios
        )

        self.q_proj = QKVProjection(heads)
        self.k_proj = QKVProjection(heads)
        self.v_proj = QKVProjection(heads)

        # PAPER_UNSPECIFIED_ASSUMPTION:
        # paper says adaptive fusion but gives no exact implementation.
        self.scale_logits = nn.Parameter(torch.zeros(len(self.k_values)))

        self.out_proj = nn.Conv2d(heads, 1, kernel_size=1, bias=False)

    @staticmethod
    def sparse_attention(scores, values, topk):
        indices = scores.topk(topk, dim=-1).indices
        mask = torch.zeros_like(scores, dtype=torch.bool)
        mask.scatter_(-1, indices, True)
        sparse_scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)
        weights = torch.softmax(sparse_scores, dim=-1)
        return weights @ values, indices

    def forward(self, x, return_indices=False):
        # [B,H,C,T]
        q = F.normalize(self.q_proj(x), p=2, dim=-1)
        k = F.normalize(self.k_proj(x), p=2, dim=-1)
        v = self.v_proj(x)

        # Paper formula uses d=C/Nh.
        d = self.channels / self.heads
        scores = (q @ k.transpose(-1, -2)) / math.sqrt(d)  # [B,H,C,C]

        branches, selections = [], []
        for topk in self.k_values:
            branch, indices = self.sparse_attention(scores, v, topk)
            branches.append(branch)
            selections.append(indices)

        alpha = torch.softmax(self.scale_logits, dim=0)
        fused = sum(w * b for w, b in zip(alpha, branches))

        # Figure 1 shows a residual path.
        out = x + self.out_proj(fused)

        if return_indices:
            return out, selections, scores
        return out

## 4. TCN branch: exact Table-1 shapes

In [21]:
class BSiLU(nn.Module):
    def __init__(self, alpha=1.67):
        super().__init__()
        self.alpha = alpha

    def forward(self, x):
        return (x + self.alpha) * torch.sigmoid(x) - self.alpha / 2.0


class PaperTCN(nn.Module):
    def __init__(self, channels, samples):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=(channels,1), bias=False)
        self.bn1 = nn.BatchNorm2d(16)

        # PAPER_INTERNAL_INCONSISTENCY:
        # text says p=23 -> PyTorch output 16;
        # table requires output 15, so use p=22.
        self.conv2 = nn.Conv2d(
            16, 16, kernel_size=(1,samples), stride=(1,3), padding=(0,22), bias=False
        )
        self.bn2 = nn.BatchNorm2d(16)

        self.conv3 = nn.Conv2d(16, 16, kernel_size=(1,12), bias=False)
        self.bn3 = nn.BatchNorm2d(16)

        self.conv4 = nn.Conv2d(16, 32, kernel_size=(1,4), bias=False)
        self.bn4 = nn.BatchNorm2d(32)

        self.act = BSiLU()

        # Paper figure/text: 32-D Conv4 feature -> FC/MLP -> 4-D branch feature.
        self.feature_fc = nn.Linear(32, 4)

    def forward(self, x, return_shapes=False):
        shapes = {}

        x = self.act(self.bn1(self.conv1(x)))
        shapes["conv1"] = tuple(x.shape)

        x = self.act(self.bn2(self.conv2(x)))
        shapes["conv2"] = tuple(x.shape)

        x = self.act(self.bn3(self.conv3(x)))
        shapes["conv3"] = tuple(x.shape)

        x = self.act(self.bn4(self.conv4(x)))
        shapes["conv4"] = tuple(x.shape)

        feature = self.feature_fc(x.flatten(1))
        shapes["feature"] = tuple(feature.shape)

        return (feature, shapes) if return_shapes else feature

## 5. FCN branch: Spatial Conv → 1×1 → FFT → KAN

In [22]:
class SplineKANLinear(nn.Module):
    # PAPER_UNSPECIFIED_ASSUMPTION:
    # paper does not provide KAN grid/order/layer/library.
    def __init__(self, in_features, out_features,
                 grid_size=KAN_GRID_SIZE, spline_order=KAN_SPLINE_ORDER,
                 grid_min=-3.0, grid_max=3.0):
        super().__init__()
        self.grid_size = grid_size
        self.spline_order = spline_order
        self.grid_min = grid_min
        self.grid_max = grid_max

        step = (grid_max - grid_min) / grid_size
        knots = (
            torch.arange(-spline_order, grid_size + spline_order + 1,
                         dtype=torch.float32) * step + grid_min
        )
        self.register_buffer("knots", knots)

        self.base = nn.Linear(in_features, out_features)
        basis_count = grid_size + spline_order
        self.spline_weight = nn.Parameter(
            torch.empty(out_features, in_features, basis_count)
        )
        nn.init.normal_(self.spline_weight, mean=0.0, std=0.02)

    def basis(self, x):
        x = self.grid_max * torch.tanh(x / self.grid_max)
        x = x.unsqueeze(-1)
        k = self.knots

        basis = ((x >= k[:-1]) & (x < k[1:])).to(x.dtype)
        eps = torch.finfo(x.dtype).eps

        for degree in range(1, self.spline_order + 1):
            left_den = (k[degree:-1] - k[:-(degree+1)]).clamp_min(eps)
            right_den = (k[degree+1:] - k[1:-degree]).clamp_min(eps)

            left = ((x - k[:-(degree+1)]) / left_den) * basis[..., :-1]
            right = ((k[degree+1:] - x) / right_den) * basis[..., 1:]
            basis = left + right

        return basis

    def forward(self, x):
        base = self.base(F.silu(x))
        spline = torch.einsum("bin,oin->bo", self.basis(x), self.spline_weight)
        return base + spline


class PaperFCN(nn.Module):
    def __init__(self, channels, samples):
        super().__init__()
        self.spatial_conv = nn.Conv2d(
            1, 32, kernel_size=(channels,1), bias=False
        )
        self.pointwise = nn.Conv2d(
            32, 16, kernel_size=1, bias=False
        )

        fft_bins = samples // 2 + 1
        self.kan = SplineKANLinear(16 * fft_bins, 4)

    def forward(self, x, return_shapes=False):
        shapes = {}

        x = self.spatial_conv(x)
        shapes["spatial_conv"] = tuple(x.shape)

        x = self.pointwise(x)
        shapes["pointwise"] = tuple(x.shape)

        spectrum = torch.abs(torch.fft.rfft(x, dim=-1))
        shapes["fft"] = tuple(spectrum.shape)

        feature = self.kan(spectrum.flatten(1))
        shapes["feature"] = tuple(feature.shape)

        return (feature, shapes) if return_shapes else feature

## 6. SATF-CNN backbone + DANN domain adversary

In [23]:
class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, strength):
        ctx.strength = strength
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.strength * grad_output, None


def grad_reverse(x, strength):
    return GradientReversalFunction.apply(x, strength)


class SATFCNNDANN(nn.Module):
    def __init__(self):
        super().__init__()
        self.pe = PaperSinusoidalPositionEncoding(EEG_CHANNELS, MODEL_SAMPLES)
        self.pesa = PaperPESA(EEG_CHANNELS, heads=3)
        self.tcn = PaperTCN(EEG_CHANNELS, MODEL_SAMPLES)
        self.fcn = PaperFCN(EEG_CHANNELS, MODEL_SAMPLES)

        # SATF-CNN time/frequency fusion -> shared 16-D representation.
        self.fusion_fc = nn.Linear(8, 16)
        self.fusion_bn = nn.BatchNorm1d(16)
        self.fusion_act = BSiLU()

        # Frequency classifier: source labels only.
        self.classifier = nn.Linear(16, NUM_CLASSES)

        # DANN domain classifier: source(test)=0, target(four)=1.
        self.domain_classifier = nn.Sequential(
            nn.Linear(16, 32),
            BSiLU(),
            nn.Linear(32, 2),
        )

    def extract_features(self, eeg, return_debug=False):
        if eeg.ndim == 3:
            eeg = eeg.unsqueeze(1)
        if eeg.ndim != 4:
            raise ValueError(f"expected [B,C,T] or [B,1,C,T], got {eeg.shape}")

        x = self.pe(eeg)

        if return_debug:
            x, selections, scores = self.pesa(x, return_indices=True)
            tfeat, tshapes = self.tcn(x, return_shapes=True)
            ffeat, fshapes = self.fcn(x, return_shapes=True)
        else:
            x = self.pesa(x)
            tfeat = self.tcn(x)
            ffeat = self.fcn(x)

        fused = torch.cat([tfeat, ffeat], dim=1)
        embedding = self.fusion_act(self.fusion_bn(self.fusion_fc(fused)))

        if return_debug:
            return embedding, {
                "pesa_output_shape": tuple(x.shape),
                "attention_shape": tuple(scores.shape),
                "topk": self.pesa.k_values,
                "selections": selections,
                "tcn": tshapes,
                "fcn": fshapes,
                "embedding_shape": tuple(embedding.shape),
            }
        return embedding

    def forward(self, eeg, grl_strength=None, return_domain=False, return_debug=False):
        if return_debug:
            embedding, debug = self.extract_features(eeg, return_debug=True)
        else:
            embedding = self.extract_features(eeg, return_debug=False)

        class_logits = self.classifier(embedding)

        domain_logits = None
        if return_domain:
            if grl_strength is None:
                raise ValueError("grl_strength is required when return_domain=True")
            domain_logits = self.domain_classifier(
                grad_reverse(embedding, float(grl_strength))
            )

        if return_debug:
            debug["logits"] = class_logits
            debug["domain_logits"] = domain_logits
            return debug
        if return_domain:
            return class_logits, domain_logits, embedding
        return class_logits


model = SATFCNNDANN().to(DEVICE)
print(model)
print("PESA top-k:", model.pesa.k_values)
print("trainable parameters:", sum(p.numel() for p in model.parameters()))

SATFCNNDANN(
  (pe): PaperSinusoidalPositionEncoding()
  (pesa): PaperPESA(
    (q_proj): QKVProjection(
      (pointwise): Conv2d(1, 3, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (depthwise): Conv2d(3, 3, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=3, bias=False)
    )
    (k_proj): QKVProjection(
      (pointwise): Conv2d(1, 3, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (depthwise): Conv2d(3, 3, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=3, bias=False)
    )
    (v_proj): QKVProjection(
      (pointwise): Conv2d(1, 3, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (depthwise): Conv2d(3, 3, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=3, bias=False)
    )
    (out_proj): Conv2d(3, 1, kernel_size=(1, 1), stride=(1, 1), bias=False)
  )
  (tcn): PaperTCN(
    (conv1): Conv2d(1, 16, kernel_size=(8, 1), stride=(1, 1), bias=False)
    (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stat

## 7. Structural forward check: SATF-CNN + DANN

In [24]:
# Tensor-shape verification only. No synthetic accuracy.
model.eval()
with torch.no_grad():
    structural_input = torch.zeros(4, EEG_CHANNELS, MODEL_SAMPLES, device=DEVICE)
    dbg = model(
        structural_input,
        grl_strength=1.0,
        return_domain=True,
        return_debug=True,
    )

print("input:", tuple(structural_input.shape))
print("PESA:", dbg["pesa_output_shape"])
print("attention:", dbg["attention_shape"])
print("top-k:", dbg["topk"])
print("TCN:", dbg["tcn"])
print("FCN:", dbg["fcn"])
print("embedding:", dbg["embedding_shape"])
print("class output:", tuple(dbg["logits"].shape))
print("domain output:", tuple(dbg["domain_logits"].shape))

assert dbg["pesa_output_shape"] == (4, 1, EEG_CHANNELS, MODEL_SAMPLES)
assert dbg["attention_shape"] == (4, 3, EEG_CHANNELS, EEG_CHANNELS)
assert dbg["tcn"]["conv2"] == (4, 16, 1, 15)
assert dbg["tcn"]["conv3"] == (4, 16, 1, 4)
assert dbg["tcn"]["conv4"] == (4, 32, 1, 1)
assert dbg["embedding_shape"] == (4, 16)
assert dbg["logits"].shape == (4, NUM_CLASSES)
assert dbg["domain_logits"].shape == (4, 2)

print("SATF-CNN + DANN shape assertions: PASS")

input: (4, 8, 372)
PESA: (4, 1, 8, 372)
attention: (4, 3, 8, 8)
top-k: (4, 5, 6)
TCN: {'conv1': (4, 16, 1, 372), 'conv2': (4, 16, 1, 15), 'conv3': (4, 16, 1, 4), 'conv4': (4, 32, 1, 1), 'feature': (4, 4)}
FCN: {'spatial_conv': (4, 32, 1, 372), 'pointwise': (4, 16, 1, 372), 'fft': (4, 16, 1, 187), 'feature': (4, 4)}
embedding: (4, 16)
class output: (4, 3)
domain output: (4, 2)
SATF-CNN + DANN shape assertions: PASS


## 8. DANN UDA training: source class CE + source/target domain CE

In [25]:
source_train_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.long),
    ),
    batch_size=SOURCE_BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)

target_train_loader = DataLoader(
    TensorDataset(torch.tensor(X_target_adapt, dtype=torch.float32)),
    batch_size=TARGET_BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED + 1),
)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=LR_PATIENCE
)


def infinite_batches(loader):
    while True:
        for batch in loader:
            yield batch


@torch.no_grad()
def predict(eeg):
    model.eval()
    x = torch.tensor(eeg, dtype=torch.float32, device=DEVICE)
    return model(x).argmax(1).cpu().numpy()


best_validation_accuracy = -1.0
best_state = None
epochs_without_improvement = 0
history = {
    "class_loss": [],
    "domain_loss": [],
    "source_train_accuracy": [],
    "domain_accuracy": [],
    "source_validation_accuracy": [],
    "grl_strength": [],
    "learning_rate": [],
}

steps_per_epoch = max(len(source_train_loader), len(target_train_loader))
total_steps = max(1, MAX_EPOCHS * steps_per_epoch - 1)
global_step = 0

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    source_iter = infinite_batches(source_train_loader)
    target_iter = infinite_batches(target_train_loader)

    class_loss_sum = 0.0
    domain_loss_sum = 0.0
    source_correct = source_total = 0
    domain_correct = domain_total = 0
    last_grl_strength = 0.0

    for _ in range(steps_per_epoch):
        source_eeg, source_labels = next(source_iter)
        (target_eeg,) = next(target_iter)

        source_eeg = source_eeg.to(DEVICE)
        source_labels = source_labels.to(DEVICE)
        target_eeg = target_eeg.to(DEVICE)

        # Standard DANN schedule: lambda(p)=2/(1+exp(-gamma*p))-1.
        progress = global_step / total_steps
        grl_strength = 2.0 / (1.0 + math.exp(-DANN_GRL_GAMMA * progress)) - 1.0
        last_grl_strength = grl_strength
        global_step += 1

        source_count = len(source_eeg)
        target_count = len(target_eeg)
        combined_eeg = torch.cat([source_eeg, target_eeg], dim=0)

        class_logits, domain_logits, _ = model(
            combined_eeg,
            grl_strength=grl_strength,
            return_domain=True,
        )

        # Frequency classification uses SOURCE labels only.
        source_class_logits = class_logits[:source_count]
        class_loss = F.cross_entropy(source_class_logits, source_labels)

        # Domain classifier sees both sessions, but no target frequency labels.
        domain_labels = torch.cat([
            torch.zeros(source_count, dtype=torch.long, device=DEVICE),
            torch.ones(target_count, dtype=torch.long, device=DEVICE),
        ])
        domain_loss = F.cross_entropy(domain_logits, domain_labels)

        loss = class_loss + DANN_DOMAIN_WEIGHT * domain_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        class_loss_sum += class_loss.item()
        domain_loss_sum += domain_loss.item()
        source_correct += (source_class_logits.argmax(1) == source_labels).sum().item()
        source_total += source_count
        domain_correct += (domain_logits.argmax(1) == domain_labels).sum().item()
        domain_total += len(domain_labels)

    source_train_accuracy = source_correct / max(source_total, 1)
    domain_accuracy = domain_correct / max(domain_total, 1)

    # Strict model selection: ONLY source internal validation labels.
    validation_prediction = predict(X_validation)
    validation_accuracy = float(np.mean(validation_prediction == y_validation))

    scheduler.step(validation_accuracy)
    lr = optimizer.param_groups[0]["lr"]

    history["class_loss"].append(class_loss_sum / steps_per_epoch)
    history["domain_loss"].append(domain_loss_sum / steps_per_epoch)
    history["source_train_accuracy"].append(source_train_accuracy)
    history["domain_accuracy"].append(domain_accuracy)
    history["source_validation_accuracy"].append(validation_accuracy)
    history["grl_strength"].append(last_grl_strength)
    history["learning_rate"].append(lr)

    if validation_accuracy > best_validation_accuracy:
        best_validation_accuracy = validation_accuracy
        best_state = {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        }
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch == 1 or epoch % 20 == 0:
        print(
            f"epoch={epoch:03d} "
            f"cls_loss={history['class_loss'][-1]:.4f} "
            f"dom_loss={history['domain_loss'][-1]:.4f} "
            f"src_train={source_train_accuracy:.4f} "
            f"domain_acc={domain_accuracy:.4f} "
            f"src_val={validation_accuracy:.4f} "
            f"grl={last_grl_strength:.3f} "
            f"lr={lr:.2e}"
        )

    if epochs_without_improvement >= EARLY_STOP_PATIENCE:
        print("early stop at epoch", epoch)
        break

assert best_state is not None
model.load_state_dict(best_state)
model.to(DEVICE)
model.eval()

print("best source validation:", best_validation_accuracy)
print("four_1~four_4 were used as UNLABELED UDA target data")
print("test_8/four_5 are still LOCKED")

epoch=001 cls_loss=1.0650 dom_loss=0.6792 src_train=0.4621 domain_acc=0.5550 src_val=0.5060 grl=0.009 lr=1.00e-03
epoch=020 cls_loss=0.0608 dom_loss=0.9164 src_train=1.0000 domain_acc=0.3000 src_val=0.5181 grl=0.197 lr=1.00e-03
epoch=040 cls_loss=0.0146 dom_loss=0.7884 src_train=1.0000 domain_acc=0.3588 src_val=0.7349 grl=0.379 lr=5.00e-04
epoch=060 cls_loss=0.0116 dom_loss=0.6542 src_train=1.0000 domain_acc=0.6825 src_val=0.6506 grl=0.537 lr=5.00e-04
epoch=080 cls_loss=0.0133 dom_loss=0.7159 src_train=1.0000 domain_acc=0.3425 src_val=0.6867 grl=0.664 lr=2.50e-04
epoch=100 cls_loss=0.0047 dom_loss=0.6865 src_train=1.0000 domain_acc=0.5475 src_val=0.6506 grl=0.761 lr=2.50e-04
epoch=120 cls_loss=0.0057 dom_loss=0.7243 src_train=1.0000 domain_acc=0.2550 src_val=0.6506 grl=0.833 lr=1.25e-04
epoch=140 cls_loss=0.0041 dom_loss=0.7018 src_train=1.0000 domain_acc=0.3962 src_val=0.6506 grl=0.885 lr=6.25e-05
early stop at epoch 152
best source validation: 0.8072289156626506
four_1~four_4 were us

## 9. Freeze DANN model, then load test_8 / four_5 once

In [26]:
for p in model.parameters():
    p.requires_grad_(False)
model.eval()
MODEL_FROZEN = True

print("model frozen; held-out labeled files unlocked")

source_test_records = load_labeled_files(TEST_FILES)
drift_test_records = load_labeled_files(DRIFT_FILES)

X_source_test, y_source_test, _ = concatenate_labeled(source_test_records)
X_drift_test, y_drift_test, _ = concatenate_labeled(drift_test_records)

model frozen; held-out labeled files unlocked
labeled four_5_ssvep original 372 shape (125, 8, 372) counts [25 44 56]
labeled test_8_ssvep original 373 shape (67, 8, 372) counts [23 24 20]


In [27]:
def report(name, labels, pred):
    correct = int(np.sum(pred == labels))
    acc = correct / len(labels)
    bal = balanced_accuracy_score(labels, pred)
    recalls = recall_score(
        labels, pred, labels=np.arange(NUM_CLASSES),
        average=None, zero_division=0
    )
    cm = confusion_matrix(labels, pred, labels=np.arange(NUM_CLASSES))

    print(f"\n{name}")
    print(f"accuracy: {acc:.4f} ({correct}/{len(labels)})")
    print(f"balanced accuracy: {bal:.4f}")
    for f, r in zip(FREQUENCIES_HZ, recalls):
        print(f"recall {f:g} Hz: {r:.4f}")
    print("confusion matrix rows=true [15,12,10], cols=pred [15,12,10]")
    print(cm)

    return acc, bal, recalls, cm

source_pred = predict(X_source_test)
drift_pred = predict(X_drift_test)

report("test_8 source diagnostic", y_source_test, source_pred)
report("four_5 final unseen-session", y_drift_test, drift_pred)


test_8 source diagnostic
accuracy: 0.7840 (98/125)
balanced accuracy: 0.8025
recall 15 Hz: 0.8800
recall 12 Hz: 0.7955
recall 10 Hz: 0.7321
confusion matrix rows=true [15,12,10], cols=pred [15,12,10]
[[22  2  1]
 [ 7 35  2]
 [ 8  7 41]]

four_5 final unseen-session
accuracy: 0.3881 (26/67)
balanced accuracy: 0.3726
recall 15 Hz: 0.8261
recall 12 Hz: 0.2917
recall 10 Hz: 0.0000
confusion matrix rows=true [15,12,10], cols=pred [15,12,10]
[[19  2  2]
 [17  7  0]
 [15  5  0]]


(0.3880597014925373,
 0.3725845410628019,
 array([0.82608696, 0.29166667, 0.        ]),
 array([[19,  2,  2],
        [17,  7,  0],
        [15,  5,  0]]))

## 10. PESA selections on final trials

In [28]:
sample = torch.tensor(X_drift_test[:3], dtype=torch.float32, device=DEVICE)
with torch.no_grad():
    dbg = model(sample, return_debug=True)

for topk, indices in zip(dbg["topk"], dbg["selections"]):
    indices = indices.cpu().numpy()
    print(f"top-{topk} shape={indices.shape}")
    for trial_idx in range(len(sample)):
        print(f"  trial {trial_idx+1}: {indices[trial_idx].tolist()}")

top-4 shape=(3, 3, 8, 4)
  trial 1: [[[1, 7, 5, 3], [7, 2, 3, 5], [7, 3, 5, 4], [7, 4, 5, 3], [7, 5, 3, 6], [7, 5, 4, 2], [7, 5, 4, 2], [7, 4, 2, 5]], [[2, 1, 4, 7], [3, 7, 6, 5], [4, 3, 5, 6], [5, 7, 0, 1], [6, 0, 5, 2], [0, 3, 2, 1], [7, 4, 5, 0], [0, 6, 2, 1]], [[0, 3, 6, 2], [0, 1, 6, 5], [0, 2, 1, 6], [0, 2, 3, 1], [0, 2, 1, 4], [0, 2, 1, 4], [0, 6, 2, 1], [0, 6, 2, 1]]]
  trial 2: [[[1, 7, 5, 3], [7, 2, 3, 5], [7, 3, 5, 4], [7, 4, 5, 3], [7, 5, 3, 6], [7, 5, 4, 2], [7, 5, 4, 2], [7, 4, 2, 5]], [[2, 1, 4, 3], [3, 7, 5, 6], [4, 7, 3, 5], [5, 7, 0, 1], [6, 0, 5, 2], [0, 3, 1, 2], [7, 4, 5, 0], [0, 6, 1, 2]], [[0, 6, 4, 2], [0, 1, 6, 5], [0, 1, 2, 6], [0, 2, 1, 3], [0, 2, 1, 4], [0, 2, 1, 4], [0, 6, 2, 1], [0, 6, 1, 2]]]
  trial 3: [[[1, 7, 5, 3], [7, 2, 3, 5], [7, 3, 5, 4], [7, 5, 4, 3], [7, 5, 6, 3], [7, 5, 4, 2], [7, 5, 4, 2], [7, 4, 2, 5]], [[2, 1, 4, 7], [3, 7, 5, 6], [4, 7, 3, 5], [5, 7, 0, 1], [6, 0, 5, 2], [0, 3, 7, 1], [7, 4, 5, 0], [0, 1, 6, 2]], [[0, 3, 4, 5], [0, 1, 6, 5]

## Reporting boundary

这版实验必须称为 **SATF-CNN + DANN UDA (test→four)**。

训练阶段实际使用：
- `test_2~test_7`：EEG + 频率标签，source classification CE；
- `four_1~four_4`：**只读取 EEG**，作为 target domain，参与 domain CE；
- GRL 让 SATF-CNN 的 16-D shared feature 对 `test/four` 域判别变困难；
- source validation 仍只来自 `test_2~test_7`，负责 scheduler / early stopping / best checkpoint。

训练阶段禁止使用：
- `four_1~four_4` 的频率标签；
- `test_8` 的 EEG/标签；
- `four_5` 的 EEG/标签。

冻结后：
- `test_8` 只做 source-session diagnostic；
- `four_5` 是 final target-session test。

所以这不是 zero-shot，而是 **two-session unsupervised domain adaptation (UDA)**。